# Deploy FastAPI Weather App to Azure Container Instance

Deploys `API_Microservices/` as a Docker container to ACI using the Azure Python SDK.
No Azure CLI installation required — authentication is reused from the azureml workspace session.

## 1. Connect to workspace and get ACR details

In [ ]:
import subprocess, time
from azure.core.credentials import AccessToken
from azureml.core import Workspace

ws = Workspace(
    subscription_id='f2f70602-65d9-479b-8014-a1c92a06ef0e',
    resource_group='Learn_MLOps',
    workspace_name='MLOps'
)

acr_name = ws.get_details()['containerRegistry'].split('/')[-1]
acr_login_server = f'{acr_name}.azurecr.io'
image_tag = f'{acr_login_server}/weather-fastapi:latest'

# Wrap azureml token so azure-mgmt SDKs can use it (no CLI needed)
class AzureMLCredential:
    def __init__(self, auth):
        self._auth = auth
    def get_token(self, *scopes, **kwargs):
        token = self._auth.get_authentication_header()['Authorization'].replace('Bearer ', '')
        return AccessToken(token, int(time.time()) + 3600)

cred = AzureMLCredential(ws._auth_object)

print(f'ACR:       {acr_login_server}')
print(f'Image tag: {image_tag}')

## 2. Get ACR credentials

In [ ]:
from azure.mgmt.containerregistry import ContainerRegistryManagementClient
from azure.mgmt.containerregistry.models import RegistryUpdateParameters

acr_client = ContainerRegistryManagementClient(cred, ws.subscription_id)

acr_client.registries.begin_update(
    'Learn_MLOps', acr_name,
    RegistryUpdateParameters(admin_user_enabled=True)
).result()

keys = acr_client.registries.list_credentials('Learn_MLOps', acr_name)
acr_username = keys.username
acr_password = keys.passwords[0].value

print(f'ACR username: {acr_username}')
print('ACR password: [retrieved]')

## 3. Docker login, build and push

In [ ]:
import os

# Login
login = subprocess.run(
    ['docker', 'login', acr_login_server, '--username', acr_username, '--password-stdin'],
    input=acr_password, capture_output=True, text=True
)
print('Login:', login.stdout.strip())

# Build — run from project root so API_Microservices/ is found
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
build = subprocess.run(
    ['docker', 'build', '-t', image_tag, 'API_Microservices/'],
    cwd=project_root, text=True
)
print('Build exit code:', build.returncode)

# Push
push = subprocess.run(['docker', 'push', image_tag], text=True)
print('Push exit code:', push.returncode)

## 4. Deploy to ACI

In [ ]:
from azure.mgmt.containerinstance import ContainerInstanceManagementClient
from azure.mgmt.containerinstance.models import (
    ContainerGroup, Container, ContainerPort, ResourceRequirements,
    ResourceRequests, ImageRegistryCredential, IpAddress, Port,
    OperatingSystemTypes, ContainerGroupIpAddressType
)

aci_client = ContainerInstanceManagementClient(cred, ws.subscription_id)

container_group = ContainerGroup(
    location='eastus2',
    containers=[
        Container(
            name='weather-fastapi',
            image=image_tag,
            resources=ResourceRequirements(
                requests=ResourceRequests(cpu=1.0, memory_in_gb=1.0)
            ),
            ports=[ContainerPort(port=80)]
        )
    ],
    image_registry_credentials=[
        ImageRegistryCredential(
            server=acr_login_server,
            username=acr_username,
            password=acr_password
        )
    ],
    ip_address=IpAddress(
        ports=[Port(protocol='TCP', port=80)],
        type=ContainerGroupIpAddressType.PUBLIC,
        dns_name_label='weather-fastapi-aci'
    ),
    os_type=OperatingSystemTypes.LINUX
)

print('Deploying... (1-2 minutes)')
result = aci_client.container_groups.begin_create_or_update(
    'Learn_MLOps', 'weather-fastapi-aci', container_group
).result()

fqdn = result.ip_address.fqdn
print(f'State:       {result.provisioning_state}')
print(f'Predict URL: http://{fqdn}/predict')
print(f'Docs URL:    http://{fqdn}/docs')

## 5. Test the endpoint

In [ ]:
import requests

url = f'http://{fqdn}/predict'
payload = {
    'temp_c': 34.927778,
    'humidity': 0.24,
    'wind_speed_kmph': 7.3899,
    'wind_bearing_degree': 83,
    'visibility_km': 16.1000,
    'pressure_millibars': 1016.51,
    'current_weather_condition': 1
}

response = requests.post(url, json=payload)
print(f'Status:     {response.status_code}')
print(f'Prediction: {response.json()}')